In [ ]:
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes unsloth_zoo

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-nn7w45jc/unsloth_479aeeae7bc9411ab9d2e6de182a3716
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-nn7w45jc/unsloth_479aeeae7bc9411ab9d2e6de182a3716
  Resolved https://github.com/unslothai/unsloth.git to commit fc861cc8703dfab892127742d4000c960689d9aa
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.7.5-py3-none-any.whl size=38839138 sha256=408f1cc12ad657cabeebd0db052523c1602302cbaa086881ab69a768ed94e157
  Stored in directory: /tmp/pip-ephem-wheel-cache-1shxfn3o/wheels/60/3e/1f/e576c07051d90cf64b6a41434d87ccf4db33fafd5343bf5de0
Successfully built unsloth
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 18.1 MB/s eta 0:00:00
   ━━━━

In [ ]:
import torch
import gc

# Delete model variables from memory
del model
del tokenizer

# Force garbage collection and flush CUDA cache
gc.collect()
torch.cuda.empty_cache()

# Verify memory is freed
print(f"Allocated GPU Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Allocated GPU Memory: 0.03 GB


In [ ]:
pip install evaluate datasets transformers scikit-learn pandas

In [ ]:
from google.colab import files
#importing th elocal csv file containing the training data
uploaded = files.upload()

Saving Suicide_Detection.csv to Suicide_Detection.csv


In [ ]:
import pandas as pd

# 1. Load CSV with robust encoding handling
# Fixed: Removed 'errors' argument which is not supported by read_csv
df = pd.read_csv("Suicide_Detection.csv", usecols=['text', 'class'], encoding='utf-8')

# 2. Fix common UTF-8 encoding corruption artifacts
df['text'] = df['text'].str.replace("â€™", "'", regex=False)
df['text'] = df['text'].str.replace("â€", '"', regex=False)
df['text'] = df['text'].str.replace("lâ€™m", "I'm", regex=False)

# 3. Clean up whitespace and truncate text
df['text'] = df['text'].astype(str).str.strip()
df['text'] = df['text'].str.split().str[:50].str.join(' ')

# 4. Create an exact 50/50 balanced sample
df_crisis = df[df['class'] == 'suicide'].sample(n=1500, random_state=42)
df_safe = df[df['class'] == 'non-suicide'].sample(n=1500, random_state=42)

df_balanced = pd.concat([df_crisis, df_safe]).sample(frac=1, random_state=42).reset_index(drop=True)

print("Cleaned Dataset Ready!")
print(df_balanced['class'].value_counts())

Cleaned Dataset Ready!
class
non-suicide    1500
suicide        1500
Name: count, dtype: int64


In [ ]:
import torch
import pandas as pd
import json
import warnings
from tqdm.notebook import tqdm
from unsloth import FastLanguageModel

warnings.filterwarnings("ignore")

# 1. Load Model with Unsloth
max_seq_length = 512
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

# 2. Use the balanced dataset from the previous cell
# df_balanced is already defined in Mw3lrlAfC2Vt

# 3. Enhanced Prompt with Clinical Rubric
def build_prompt(text_sample):
    rubric = """
    Level 0: No risk. Casual chat.
    Level 1: Mild. Stress or sadness but no self-harm intent.
    Level 2: Moderate. Persistent hopelessness or ideation without a plan.
    Level 3: High. Strong intent, specific thoughts of methods.
    Level 4: Emergency. Imminent threat, goodbye notes, or immediate plan.
    """
    messages = [
        {"role": "system", "content": f"You are a clinical psychologist. Analyze the risk level based on this rubric: {rubric}. Output ONLY valid JSON."},
        {"role": "user", "content": f"Analyze this text: '{text_sample}'\nReturn JSON: {{'risk_level': int, 'hopelessness': 0 or 1, 'isolation': 0 or 1}}"}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# 4. Batched Inference
BATCH_SIZE = 16
texts = df_balanced['text'].tolist()
prompts = [build_prompt(t) for t in texts]
annotations = []

for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc="Distilling Balanced Data"):
    batch_prompts = prompts[i:i + BATCH_SIZE]
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=max_seq_length).to("cuda")

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=64, do_sample=False)

    decoded = tokenizer.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)
    for res in decoded:
        try:
            start = res.find('{')
            end = res.rfind('}') + 1
            annotations.append(json.loads(res[start:end]))
        except:
            annotations.append({"risk_level": 0, "hopelessness": 0, "isolation": 0})

df_distilled = pd.concat([df_balanced.reset_index(drop=True), pd.DataFrame(annotations)], axis=1)
df_distilled.to_csv("distilled_dataset.csv", index=False)
print("Balanced distillation complete! Distilled labels saved to 'distilled_dataset.csv'.")
print(df_distilled['risk_level'].value_counts())

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1432: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.5: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Distilling Balanced Data:   0%|          | 0/188 [00:00<?, ?it/s]

Balanced distillation complete! Distilled labels saved to 'distilled_dataset.csv'.
risk_level
0    1300
1     737
4     369
2     314
3     280
Name: count, dtype: int64


In [ ]:
import os
import numpy as np
import pandas as pd
import evaluate
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# Disable wandb to avoid interactive prompts
os.environ["WANDB_DISABLED"] = "true"

# 1. Load the balanced distilled dataset
df = pd.read_csv("distilled_dataset.csv")
df['risk_level'] = df['risk_level'].fillna(0).astype(int).clip(0, 4)

# Convert to HF Dataset
dataset = Dataset.from_pandas(df[['text', 'risk_level']].rename(columns={'risk_level': 'label'}))

# 2. Tokenize
student_id = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(student_id)

def tokenize(batch):
    return tokenizer(batch['text'], padding="max_length", truncation=True, max_length=128)

tokenized_ds = dataset.map(tokenize, batched=True)
split_ds = tokenized_ds.train_test_split(test_size=0.1, seed=42)

# 3. Load Model
model = AutoModelForSequenceClassification.from_pretrained(student_id, num_labels=5)

# 4. Metrics
accuracy_metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

# 5. Training Args
args = TrainingArguments(
    output_dir="./student_minilm",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=5,
    learning_rate=5e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
    logging_steps=10,
    report_to=None
)

# 6. Train
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=split_ds['train'],
    eval_dataset=split_ds['test'],
    compute_metrics=compute_metrics
)

trainer.train()
model.save_pretrained("./final_minilm")
tokenizer.save_pretrained("./final_minilm")
print("Re-training complete with balanced data!")

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at sentence-transformers/all-MiniLM-L6-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,700 | Num Epochs = 5 | Total steps = 425
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 1 x 1) = 32
 "-____-"     Trainable parameters = 22,715,141 of 22,715,141 (100.00% trained)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.999000,0.964385,0.593333
2,0.823100,0.857830,0.663333
3,0.726900,0.792951,0.696667
4,0.594700,0.786279,0.683333
5,0.463300,0.777412,0.696667


Re-training complete with balanced data!


In [ ]:
!optimum-cli export onnx --model ./final_minilm --task text-classification ./onnx_minilm_deploy/ --overwrite

`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


In [ ]:
import numpy as np
from transformers import AutoTokenizer
import onnxruntime as ort
import os

# Reload the tokenizer and ONNX session to pick up the new weights
tokenizer = AutoTokenizer.from_pretrained("./final_minilm")
onnx_path = "./onnx_minilm_deploy/model.onnx"
ort_session = ort.InferenceSession(onnx_path)

ACTION_MAPPING = {
    0: "SAFE: Allow request normally.",
    1: "MILD: Log search query.",
    2: "MODERATE: Display self-care resources.",
    3: "HIGH: Alert admin dashboard.",
    4: "EMERGENCY: Intercept DNS & show crisis helpline."
}

def analyze_query(query_text):
    inputs = tokenizer(query_text, return_tensors="np", max_length=128, padding="max_length", truncation=True)
    onnx_inputs = {
        'input_ids': inputs['input_ids'].astype(np.int64),
        'attention_mask': inputs['attention_mask'].astype(np.int64),
        'token_type_ids': inputs['token_type_ids'].astype(np.int64)
    }
    logits = ort_session.run(None, onnx_inputs)[0]
    probabilities = np.exp(logits) / np.sum(np.exp(logits), axis=-1, keepdims=True)
    predicted_class = int(np.argmax(probabilities))
    confidence = float(probabilities[0][predicted_class])

    return {
        "query": query_text,
        "risk_level": predicted_class,
        "confidence": round(confidence, 4),
        "action": ACTION_MAPPING[predicted_class]
    }

# Final validation
print("--- Testing Results ---")
print(analyze_query("how to bake chocolate chip cookies"))
print(analyze_query("I feel hopeless and want to end everything"))

--- Testing Results ---
{'query': 'how to bake chocolate chip cookies', 'risk_level': 0, 'confidence': 0.9335, 'action': 'SAFE: Allow request normally.'}
{'query': 'I feel hopeless and want to end everything', 'risk_level': 3, 'confidence': 0.37, 'action': 'HIGH: Alert admin dashboard.'}


In [ ]:
from google.colab import files
import shutil

# Zip the exported folder
shutil.make_archive("onnx_minilm_deploy", "zip", "./onnx_minilm_deploy")

# Download zip file
files.download("onnx_minilm_deploy.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>